# 🏎️ Fabric Racing Game - Race Simulator

This notebook simulates a racing game with 4 players, generating telemetry events in real-time.

**Events Generated:**
- RaceStart, RaceEnd
- LapComplete
- Position updates (every 100ms)
- SpeedBoost, Collision, Overtake

In [ ]:
# Configuration - Update these values
EVENTSTREAM_ENDPOINT = "<YOUR_CUSTOM_ENDPOINT_URL>"
RACE_DURATION_SECONDS = 60
NUM_LAPS = 3
EVENT_INTERVAL_MS = 100

In [ ]:
import json
import uuid
import time
import random
import math
from datetime import datetime
from typing import List, Dict
import requests

In [ ]:
# Car configurations
CARS = [
    {"id": 1, "player": "Player1", "color": "Red", "max_speed": 180},
    {"id": 2, "player": "Player2", "color": "Blue", "max_speed": 175},
    {"id": 3, "player": "Player3", "color": "Green", "max_speed": 185},
    {"id": 4, "player": "Player4", "color": "Yellow", "max_speed": 170}
]

# Track sections (oval track)
TRACK_SECTIONS = [
    "StartFinish", "Turn1", "Backstraight", "Turn2"
]

# Track length in units
TRACK_LENGTH = 1000

In [ ]:
class RaceSimulator:
    """Simulates a racing game with 4 players"""
    
    def __init__(self, endpoint: str):
        self.endpoint = endpoint
        self.session_id = str(uuid.uuid4())
        self.cars = {}
        self._init_cars()
    
    def _init_cars(self):
        """Initialize car positions"""
        for car in CARS:
            self.cars[car['id']] = {
                **car,
                'position': car['id'] * 10,  # Staggered start
                'speed': 0,
                'lap': 0,
                'x': 0,
                'y': 0
            }
    
    def _get_section(self, position: float) -> str:
        """Get track section from position"""
        section_length = TRACK_LENGTH / len(TRACK_SECTIONS)
        idx = int((position % TRACK_LENGTH) / section_length)
        return TRACK_SECTIONS[idx]
    
    def _calculate_xy(self, position: float) -> tuple:
        """Convert linear position to X,Y on oval track"""
        angle = (position % TRACK_LENGTH) / TRACK_LENGTH * 2 * math.pi
        x = 500 + 300 * math.cos(angle)
        y = 300 + 200 * math.sin(angle)
        return round(x, 2), round(y, 2)
    
    def send_event(self, event: Dict):
        """Send event to Eventstream"""
        try:
            response = requests.post(
                self.endpoint,
                headers={"Content-Type": "application/json"},
                json=event,
                timeout=5
            )
            if response.status_code >= 400:
                print(f"Error sending event: {response.status_code}")
        except Exception as e:
            print(f"Failed to send event: {e}")
    
    def create_event(self, player_id: str, event_type: str, car: Dict) -> Dict:
        """Create a telemetry event"""
        x, y = self._calculate_xy(car['position'])
        return {
            "EventId": str(uuid.uuid4()),
            "Timestamp": datetime.utcnow().isoformat() + "Z",
            "PlayerId": player_id,
            "EventType": event_type,
            "CarId": car['id'],
            "PositionX": x,
            "PositionY": y,
            "Speed": round(car['speed'], 1),
            "LapNumber": car['lap'],
            "TrackSection": self._get_section(car['position']),
            "GameSessionId": self.session_id
        }
    
    def update_car(self, car: Dict, dt: float):
        """Update car position and speed"""
        # Accelerate/decelerate with some randomness
        target_speed = car['max_speed'] * (0.7 + random.random() * 0.3)
        
        # Slow down in turns
        section = self._get_section(car['position'])
        if 'Turn' in section:
            target_speed *= 0.7
        
        # Smooth acceleration
        car['speed'] += (target_speed - car['speed']) * 0.1
        
        # Update position
        old_lap = car['position'] // TRACK_LENGTH
        car['position'] += car['speed'] * dt
        new_lap = car['position'] // TRACK_LENGTH
        
        # Check for lap completion
        if new_lap > old_lap:
            car['lap'] = int(new_lap)
            return True  # Lap completed
        return False
    
    def run_race(self, duration_seconds: int = 60, num_laps: int = 3):
        """Run a complete race simulation"""
        print(f"🏁 Starting race - Session: {self.session_id[:8]}...")
        
        # Send RaceStart events
        for car in self.cars.values():
            event = self.create_event(car['player'], "RaceStart", car)
            self.send_event(event)
        
        start_time = time.time()
        tick = 0
        
        while True:
            elapsed = time.time() - start_time
            
            # Check if race should end
            leader_lap = max(c['lap'] for c in self.cars.values())
            if leader_lap >= num_laps or elapsed >= duration_seconds:
                break
            
            # Update each car
            for car in self.cars.values():
                lap_completed = self.update_car(car, 0.1)
                
                # Send position update every tick
                event = self.create_event(car['player'], "Position", car)
                self.send_event(event)
                
                # Send lap complete event
                if lap_completed:
                    lap_event = self.create_event(car['player'], "LapComplete", car)
                    self.send_event(lap_event)
                    print(f"  🏁 {car['player']} completed lap {car['lap']}")
                
                # Random events
                if random.random() < 0.01:
                    boost_event = self.create_event(car['player'], "SpeedBoost", car)
                    self.send_event(boost_event)
            
            tick += 1
            if tick % 50 == 0:
                print(f"  ⏱️ {elapsed:.1f}s - Leader on lap {leader_lap}")
            
            time.sleep(EVENT_INTERVAL_MS / 1000)
        
        # Send RaceEnd events
        positions = sorted(self.cars.values(), 
                          key=lambda c: (c['lap'], c['position']), 
                          reverse=True)
        
        print("\n🏆 RACE RESULTS:")
        for i, car in enumerate(positions):
            event = self.create_event(car['player'], "RaceEnd", car)
            event['Position'] = i + 1
            self.send_event(event)
            medal = ["🥇", "🥈", "🥉", "4️⃣"][i]
            print(f"  {medal} {car['player']} ({car['color']}) - Lap {car['lap']}")
        
        print(f"\n✅ Race complete! {tick} position events sent.")

In [ ]:
# Run the race!
if EVENTSTREAM_ENDPOINT.startswith("<"):
    print("⚠️ Please update EVENTSTREAM_ENDPOINT with your Custom Endpoint URL")
else:
    simulator = RaceSimulator(EVENTSTREAM_ENDPOINT)
    simulator.run_race(
        duration_seconds=RACE_DURATION_SECONDS,
        num_laps=NUM_LAPS
    )

## 📊 Verify Data in KQL

Run these queries in the KQL Database to analyze your race:

```kql
// Count events by type
GameEvents
| where GameSessionId == "<your-session-id>"
| summarize count() by EventType

// Lap times by player
GameEvents
| where EventType == "LapComplete"
| summarize LapTime = max(Timestamp) by PlayerId, LapNumber
| order by LapNumber asc, LapTime asc

// Speed heatmap
GameEvents
| where EventType == "Position"
| summarize AvgSpeed = avg(Speed) by TrackSection, PlayerId
| render columnchart
```